In [31]:
# Allowing dynamic links, hot-reload, to our custom modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import os, re, sys, traceback, csv, random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sentencepiece as spm

from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data

os.getcwd()

'c:\\Users\\shrey\\Downloads\\dl-gen-lyrics\\sandbox'

In [33]:
import generator_core as core
from aspect_teal import Teal
import aspect_teal
from aspect_midnight import Word2Vec_SkipGram, ArrayToDatasetForW2V, ConditionalDataset

In [34]:
from dl_trainer import Trainer

In [35]:
batch_count = 103
genres = {'rap', 'rb', 'pop', 'country', 'rock', 'misc'}
print(genres)
print(batch_count)

{'misc', 'rock', 'rap', 'pop', 'rb', 'country'}
103


In [36]:
total_subset_size = 100000
total_group_size = total_subset_size // len(genres) + 1
max_songs_per_artist = 50

In [37]:
ds_subset = pd.read_csv(os.path.join('temp', 'genius_lyrics_subset.csv'))
display(ds_subset.head())

,Unnamed: 0,title,tag,artist,year,lyrics,id,language
0,123725,In Memory of W. B. Yeats,misc,W. H. Auden,2013,I\n\nHe disappeared in the dead of winter:\nTh...,134446,en
1,2614099,Sthlm City,pop,Rebecca & Fiona,2018,"[Intro]\nI mean, like, it's okay\nOne day, you...",3931557,en
2,2550228,Fallin 2010,pop,Mozhdah,2010,[Verse 1]\nCan you take it easy\nWhile I make ...,3840000,en
3,3539706,Can the Circle Be Unbroken,country,Johnny Cash,1982,I was standing by the window\nOn one cold and ...,5350667,en
4,2650114,Marijuana,pop,Jah Cure,2019,[Intro: Jah Cure]\nOne puff one draw\nFrees th...,3982786,en


In [38]:
teal = Teal(ds_subset)

Loaded Cache for Teal._prepare_ds_data.cached
Loaded Cache for Teal._get_custom_tokens.cached
Loaded Cache for Teal._get_genre_dict.cached
Loaded Cache for Teal._prepare_tfidf.cached


C:\Users\shrey\AppData\Roaming\Python\Python314\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\shrey\AppData\Roaming\Python\Python314\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loaded Cache for Midnight._prepare_vocabulary temp\lyrics_sp.model
Loaded Cache for Teal._prepare_embedder.cached


In [41]:
teal.embedder.embeddings

Embedding(16000, 128)

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [43]:
dataset = ConditionalDataset(teal)
dataloader = data.DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=dataset.collate_fn,
)

In [53]:
model = aspect_teal.TransformerModel(teal.vocabulary.vocab_size(), {
    "d_model":128,
    "n_heads":2,
    "n_groups":2,
    "n_layers":4,
    "pe": "sinusoidal",
    "norm": "layernorm",
    "attn": "gqa",
    "ffn": "gelu"
}, teal.embedder.embeddings.weight)

In [54]:
def model_train_step(model, data): 
    return model(data[0])

def model_criteria_step(criterion, preds, truth):
    # print(preds.shape, truth.shape)
    preds = preds.permute(0, 2, 1)
    return criterion(preds, truth)


trainer = Trainer(
    model=model,
    train_dataloader=dataloader,
    epochs=1,
    # dataset_fraction=1,
    model_train_step=model_train_step,
    model_criteria_step=model_criteria_step,
)

In [55]:
trainer.train()
# CPU Time: 34 sit

Epoch:  0/ 1    Train Loss: 9.86    TTS: 173.85    ETA: 1086538.70
Epoch:  0/ 1    Train Loss: 9.08    TTS: 214.10    ETA: 668953.27
Epoch:  0/ 1    Train Loss: 8.52    TTS: 277.97    ETA: 578922.82
Epoch:  0/ 1    Train Loss: 8.12    TTS: 348.76    ETA: 544674.70


KeyboardInterrupt: 

In [30]:
"hi"

'hi'